# Cross-Pipeline Validation (Harmonized)

Computes all metrics across pipeline pairs:
- ARI, NMI (global clustering)
- Dice (per-cluster overlap)
- Jaccard + Spearman ρ (native DE + standardized DE)
- Module score profile correlation (biology preservation)

**Run AFTER** all pipeline notebooks have completed.

Uses Hungarian matching for optimal cluster correspondence.
Standardized DE isolates cluster-label drift from DE-engine drift.

In [1]:
# === CHANGE THIS TO SWITCH DATASETS ===
DATASET = "pbmc3k"   # or "lung65k"

CONFIG_PATH = "benchmark_config.json"

In [2]:
# The validation script is complex enough to run as-is.
# Execute it via subprocess so it uses argparse correctly.
import subprocess
result = subprocess.run(
    ["python", "validate_cross_pipeline_harmonized.py",
     "--dataset", DATASET, "--config", CONFIG_PATH],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr)

Validation — PBMC3k
Pipelines detected:
  scanpy_cpu   write/pbmc3k_scanpy_cpu_harmonized_clusters.csv
  rsc_gpu_0141 write/pbmc3k_rsc_gpu_0141_harmonized_clusters.csv
  seurat_cpu   write/pbmc3k_seurat_cpu_harmonized_clusters.csv
  rsc_gpu_015  write/pbmc3k_rsc_gpu_015_harmonized_clusters.csv

Summary:
dataset                  comparison  common_cells  clusters_a  clusters_b      ARI      NMI  mean_dice  mean_native_deg_jaccard  mean_native_deg_spearman  mean_standardized_deg_jaccard  mean_standardized_deg_spearman  mean_module_profile_rho
 PBMC3k  scanpy_cpu vs rsc_gpu_0141          2638           9           9 0.922560 0.934597   0.979719                 0.942537                  0.983240                       0.942537                        0.983240                 1.000000
 PBMC3k    scanpy_cpu vs seurat_cpu          2638           9          10 0.655449 0.761997   0.789819                 0.443738                  0.877465                       0.736301                        0.8

In [3]:
# Load and display the summary
import json
import pandas as pd

with open(CONFIG_PATH) as f:
    dcfg = json.load(f)["datasets"][DATASET]
prefix = dcfg["pipeline_prefix"]
val_dir = f"write/validation_{prefix}_harmonized"

summary_csv = f"{val_dir}/{prefix}_validation_summary.csv"
try:
    df = pd.read_csv(summary_csv)
    print(df.to_string(index=False))
except FileNotFoundError:
    print(f"Summary not found: {summary_csv}")

dataset                  comparison  common_cells  clusters_a  clusters_b      ARI      NMI  mean_dice  mean_native_deg_jaccard  mean_native_deg_spearman  mean_standardized_deg_jaccard  mean_standardized_deg_spearman  mean_module_profile_rho
 PBMC3k  scanpy_cpu vs rsc_gpu_0141          2638           9           9 0.922560 0.934597   0.979719                 0.942537                  0.983240                       0.942537                        0.983240                 1.000000
 PBMC3k    scanpy_cpu vs seurat_cpu          2638           9          10 0.655449 0.761997   0.789819                 0.443738                  0.877465                       0.736301                        0.829671                 0.980159
 PBMC3k   scanpy_cpu vs rsc_gpu_015          2638           9           9 0.922560 0.934597   0.979719                 0.942537                  0.983240                       0.942537                        0.983240                 1.000000
 PBMC3k  rsc_gpu_0141 vs seurat_

In [4]:
# Load detailed results
detail_json = f"{val_dir}/{prefix}_validation_details.json"
try:
    with open(detail_json) as f:
        details = json.load(f)
    for pair, metrics in details.items():
        print(f"\n=== {pair} ===")
        print(f"  ARI: {metrics['ARI']:.4f}")
        print(f"  NMI: {metrics['NMI']:.4f}")
        print(f"  Mean Dice: {metrics.get('mean_dice', 'N/A')}")
        print(f"  Mean Std DEG Jaccard: {metrics.get('mean_standardized_deg_jaccard', 'N/A')}")
        print(f"  Mean Std DEG Spearman: {metrics.get('mean_standardized_deg_spearman', 'N/A')}")
        print(f"  Mean Module ρ: {metrics.get('mean_module_profile_rho', 'N/A')}")
except FileNotFoundError:
    print(f"Details not found: {detail_json}")


=== scanpy_cpu__vs__rsc_gpu_0141 ===
  ARI: 0.9226
  NMI: 0.9346
  Mean Dice: 0.9797192565274843
  Mean Std DEG Jaccard: 0.9425367484223381
  Mean Std DEG Spearman: 0.9832404910461902
  Mean Module ρ: 1.0

=== scanpy_cpu__vs__seurat_cpu ===
  ARI: 0.6554
  NMI: 0.7620
  Mean Dice: 0.7898188286306885
  Mean Std DEG Jaccard: 0.7363011973218674
  Mean Std DEG Spearman: 0.8296707051929233
  Mean Module ρ: 0.9801587301587303

=== scanpy_cpu__vs__rsc_gpu_015 ===
  ARI: 0.9226
  NMI: 0.9346
  Mean Dice: 0.9797192565274843
  Mean Std DEG Jaccard: 0.9425367484223381
  Mean Std DEG Spearman: 0.9832404910461902
  Mean Module ρ: 1.0

=== rsc_gpu_0141__vs__seurat_cpu ===
  ARI: 0.6387
  NMI: 0.7503
  Mean Dice: 0.7819603317739764
  Mean Std DEG Jaccard: 0.7246941896940297
  Mean Std DEG Spearman: 0.8214609368082422
  Mean Module ρ: 0.9801587301587301

=== rsc_gpu_0141__vs__rsc_gpu_015 ===
  ARI: 1.0000
  NMI: 1.0000
  Mean Dice: 1.0
  Mean Std DEG Jaccard: 1.0
  Mean Std DEG Spearman: 1.0
  Mean M